# Masiez Kamikaze FPV Drone Robot — LiveKit Prototyping & Kaggle Training Setup

The simulated kamikaze FPV drone acts directly as a **LiveKit Robot** (`identity="drone-sim"` in room `"drone"`) following [LiveKit Robotics](https://docs.livekit.io/robotics/) conventions.
Instead of routing or processing heavy camera frames through the backend server, this Jupyter notebook connects directly to the **LiveKit SFU** via WebRTC (`livekit.rtc`) to:
1. **Subscribe to the live WebRTC video stream** (`drone-fpv`) as NumPy RGB arrays.
2. **Receive real-time telemetry & kamikaze mission state** (`drone:telemetry` & `drone:kamikaze_strike`) over the LiveKit DataChannel.
3. **Send robot control commands** (`teleop:command`: e.g., `deploy_next_drone`, `restart_mission`) and **vision annotations** (`ai:vision_annotations`) back to the drone.
4. **Collect frame + telemetry + mission datasets** ready for model training on **Kaggle**.

In [ ]:
import asyncio
import io
import json
import os
import time
from pathlib import Path

import numpy as np
from PIL import Image
from livekit import api, rtc
from IPython.display import display, Image as IPyImage, clear_output

LIVEKIT_URL = os.environ.get("LIVEKIT_URL", "ws://livekit:7880")
LIVEKIT_API_KEY = os.environ.get("LIVEKIT_API_KEY", "devkey")
LIVEKIT_API_SECRET = os.environ.get("LIVEKIT_API_SECRET", "devsecret_devsecret_devsecret_1234")
LIVEKIT_ROOM = os.environ.get("LIVEKIT_ROOM", "drone")
JUPYTER_IDENTITY = "jupyter-controller"

def create_livekit_token(identity: str = JUPYTER_IDENTITY) -> str:
    """Mint a LiveKit JWT token for the Jupyter robot controller/trainer."""
    return (
        api.AccessToken(LIVEKIT_API_KEY, LIVEKIT_API_SECRET)
        .with_identity(identity)
        .with_grants(
            api.VideoGrants(
                room_join=True,
                room=LIVEKIT_ROOM,
                can_publish=False,
                can_subscribe=True,
                can_publish_data=True,
            )
        )
        .to_jwt()
    )

## 1. Connect to the Drone Robot via LiveKit
Connects to the `drone` room, subscribes to the drone's camera video stream and telemetry/mission data channel, and keeps the latest RGB frame, telemetry, and kamikaze mission stats (`targetsHit`, `totalTargets`, `dronesRemaining`, `totalDrones`, `score`) in memory.

In [ ]:
class DroneRobotClient:
    def __init__(self):
        self.room = rtc.Room()
        self.latest_frame: np.ndarray | None = None
        self.latest_telemetry: dict | None = None
        self.latest_mission: dict | None = None
        self._video_task: asyncio.Task | None = None

        @self.room.on("track_subscribed")
        def _on_track(track: rtc.Track, pub: rtc.RemoteTrackPublication, participant: rtc.RemoteParticipant):
            if track.kind == rtc.TrackKind.KIND_VIDEO:
                print(f"[LiveKit] Subscribed to video track '{pub.name}' from '{participant.identity}'")
                self._video_task = asyncio.create_task(self._consume_video(track))

        @self.room.on("data_received")
        def _on_data(data_packet: rtc.DataPacket):
            try:
                msg = json.loads(bytes(data_packet.data).decode("utf-8"))
                if msg.get("type") == "drone:telemetry":
                    self.latest_telemetry = msg.get("telemetry")
                    self.latest_mission = msg.get("mission")
                elif msg.get("type") == "drone:kamikaze_strike":
                    self.latest_mission = msg.get("mission")
                    print(f"[Kamikaze Strike] Hit={msg.get('hitTarget')} | Destroyed={msg.get('destroyedTargets')} | Mission={self.latest_mission}")
            except Exception:
                pass

    async def _consume_video(self, track: rtc.Track):
        stream = rtc.VideoStream(track)
        async for event in stream:
            rgb = event.frame.convert(rtc.VideoBufferType.RGB24)
            self.latest_frame = np.frombuffer(rgb.data, dtype=np.uint8).reshape((rgb.height, rgb.width, 3)).copy()

    async def connect(self):
        await self.room.connect(LIVEKIT_URL, create_livekit_token())
        print(f"[LiveKit] Connected to {LIVEKIT_URL} (room='{LIVEKIT_ROOM}')")
        # Attach to any video track already published in the room
        for participant in self.room.remote_participants.values():
            for pub in participant.track_publications.values():
                if pub.track and pub.track.kind == rtc.TrackKind.KIND_VIDEO:
                    self._video_task = asyncio.create_task(self._consume_video(pub.track))

    async def send_command(self, cmd_type: str, **params):
        """Send a teleop command directly to the drone robot over LiveKit DataChannel."""
        payload = {
            "type": "teleop:command",
            "command": {"type": cmd_type, "params": params},
            "timestamp": int(time.time() * 1000),
        }
        await self.room.local_participant.publish_data(
            json.dumps(payload).encode("utf-8"), reliable=True, topic="teleop"
        )

    async def publish_detections(self, detections: list[dict], width: int, height: int):
        """Send AI bounding-box predictions back to the drone HUD over LiveKit DataChannel."""
        payload = {
            "type": "ai:vision_annotations",
            "detections": detections,
            "width": width,
            "height": height,
            "timestamp": int(time.time() * 1000),
        }
        await self.room.local_participant.publish_data(
            json.dumps(payload).encode("utf-8"), reliable=True
        )

    async def disconnect(self):
        if self._video_task:
            self._video_task.cancel()
        await self.room.disconnect()

drone = DroneRobotClient()
await drone.connect()

## 2. Watch Live WebRTC Stream from the Drone Robot

In [ ]:
async def watch_livekit_stream(max_frames: int = 60, scale: float = 0.5, fps: float = 10.0):
    seen = 0
    interval = 1.0 / fps
    while seen < max_frames:
        if drone.latest_frame is not None:
            img = Image.fromarray(drone.latest_frame)
            if scale != 1.0:
                img = img.resize((int(img.width * scale), int(img.height * scale)))
            buf = io.BytesIO()
            img.save(buf, format="JPEG")
            clear_output(wait=True)
            display(IPyImage(data=buf.getvalue()))
            seen += 1
        await asyncio.sleep(interval)

await watch_livekit_stream(max_frames=60)

## 3. Collect Dataset for Kaggle Training
Captures synchronized WebRTC camera frames, telemetry, and kamikaze mission state snapshots into `./dataset/` so they can be uploaded to **Kaggle** for training vision or control models.

In [ ]:
async def collect_kaggle_dataset(num_samples: int = 50, interval_sec: float = 0.5, out_dir: str = "./dataset"):
    out_path = Path(out_dir)
    frames_dir = out_path / "images"
    frames_dir.mkdir(parents=True, exist_ok=True)
    records = []

    for idx in range(num_samples):
        if drone.latest_frame is not None:
            img_name = f"frame_{idx:05d}.jpg"
            Image.fromarray(drone.latest_frame).save(frames_dir / img_name, quality=90)
            records.append({
                "image": img_name,
                "timestamp": time.time(),
                "telemetry": drone.latest_telemetry,
                "mission": drone.latest_mission,
            })
        await asyncio.sleep(interval_sec)

    with open(out_path / "metadata.jsonl", "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec) + "\n")

    print(f"Saved {len(records)} samples to {out_path.resolve()}")